# **Carbon Emission Compliance Optimizer**

## Problem Defination

Using the Carbon Majors Emission dataset from kaggle, developed a predictive model which estimates how much an industry will exceed emission limits and suggest whether they should reduce emissions or buy credits to meet govt criteria.

## Importing Required Liabraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report
import xgboost as xgb
import lightgbm as lgb
from scipy.optimize import linprog

# Dataset Overview


 **Dataset Details 1**


*  **Dataset Name :** emissions_high_granularity
*   **Source :** https://www.kaggle.com/datasets/joebeachcapital/carbon-majors-emissions-data/data
*   **File Format :** .csv


**Dataset Description**

The Carbon Majors Emissions Data contains historical greenhouse gas emissions attributed to major fossil fuel and cement producers worldwide. It combines operational and product-use emissions from investor-owned companies, state-owned enterprises, and nation-states. Each row represents the reported or estimated emissions for a specific entity in a given year.

**Feature Description**


*   Year : The reporting year of emissions data

*   Entity : Name of the company or nation-state responsible for emissions

*   Entity Type : Classification of the entity (Investor-Owned, State-Owned, Nation-State)


*   Country : Country where the entity is headquartered or operates
*   Scope 1 Emissions : Direct operational emissions (e.g., flaring, venting, methane leakage) in MtCO₂e

*   Scope 3 Emissions : Indirect emissions from combustion of sold products in MtCO₂e

*   Total Emissions : Combined Scope 1 and Scope 3 emissions in MtCO₂e

*   Cumulative Emissions : Running total of historical emissions attributed to the entity

*   Fuel Type : Type of fossil fuel or cement associated with emissions (Coal, Oil, Gas, Cement, etc.)



 **Dataset Details 2**


*  **Dataset Name :** emissions_low_granularity
*   **Source :** https://www.kaggle.com/datasets/joebeachcapital/carbon-majors-emissions-data/data
*   **File Format :** .csv


**Dataset Description**

The Carbon Majors Emissions Data contains historical greenhouse gas emissions attributed to major fossil fuel and cement producers worldwide. It combines operational and product-use emissions from investor-owned companies, state-owned enterprises, and nation-states. Each row represents the reported or estimated emissions for a specific entity in a given year.

**Feature Description**

*   YEAR : The reporting year of the data point

*   PARENT_ENTITY : The company, state-owned entity, or nation-state responsible for emissions

*   PARENT_TYPE : The type of entity – investor-owned company, state-owned entity, or nation-state

*   COMMODITY : The fuel/commodity associated with production (Oil, Natural Gas, Coal types, or Cement)

*   PRODUCTION_VALUE : The quantity of commodity produced in the reporting year

*   PRODUCTION_UNIT : The unit of measurement for production (Million barrels, Billion cubic feet, or Million tonnes depending on commodity)

*   SCOPE1_EMISSIONS_MtCO2e : Direct operational emissions from production activities (in million tonnes of CO₂ equivalent)

*   SCOPE3_EMISSIONS_MtCO2e : Indirect emissions from combustion/use of sold products (in million tonnes of CO₂ equivalent)

*   TOTAL_EMISSIONS_MtCO2e : The total greenhouse gas emissions attributed to the entity in that year (Scope 1 + Scope 3)

*   CUMULATIVE_EMISSIONS_MtCO2e : Running historical total of all emissions attributed to the entity up to that year


 **Dataset Details 3**


*  **Dataset Name :** emissions_medium_granularity
*   **Source :** https://www.kaggle.com/datasets/joebeachcapital/carbon-majors-emissions-data/data
*   **File Format :** .csv


**Dataset Description**

The Carbon Majors Emissions Data contains historical greenhouse gas emissions attributed to major fossil fuel and cement producers worldwide. It combines operational and product-use emissions from investor-owned companies, state-owned enterprises, and nation-states. Each row represents the reported or estimated emissions for a specific entity in a given year.

**Feature Description**


*   Year : The reporting year of emissions data

*   Entity : Name of the company or nation-state responsible for emissions

*   Entity Type : Classification of the entity (Investor-Owned, State-Owned, Nation-State)


*   Country : Country where the entity is headquartered or operates
*   Scope 1 Emissions : Direct operational emissions (e.g., flaring, venting, methane leakage) in MtCO₂e

*   Scope 3 Emissions : Indirect emissions from combustion of sold products in MtCO₂e

*   Total Emissions : Combined Scope 1 and Scope 3 emissions in MtCO₂e

*   Cumulative Emissions : Running total of historical emissions attributed to the entity

*   Fuel Type : Type of fossil fuel or cement associated with emissions (Coal, Oil, Gas, Cement, etc.)





## Loading the Dataset


In [ ]:
import pandas as pd
high_emission_data = pd.read_csv("/content/sample_data/emissions_high_granularity.csv")
low_emission_data = pd.read_csv("/content/sample_data/emissions_high_granularity.csv")
medium_emission_data = pd.read_csv("/content/sample_data/emissions_high_granularity.csv")

## Explore and Understand the Data

**HIGH GRANULARITY DATA**

In [ ]:
high_emission_data.head()
high_emission_data.info()
high_emission_data.describe()

**LOW GRANULARITY DATA**

In [ ]:
low_emission_data.head()
low_emission_data.info()
low_emission_data.describe()

**MEDIUM GRANULARITY DATA**

In [ ]:
medium_emission_data.head()
medium_emission_data.info()
medium_emission_data.describe()

In [14]:
import pandas as pd

df = pd.read_csv("/content/sample_data/emissions_medium_granularity.csv")
print(df.info())
print(df.isnull().sum())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12551 entries, 0 to 12550
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    12551 non-null  int64  
 1   parent_entity           12551 non-null  object 
 2   parent_type             12551 non-null  object 
 3   commodity               12551 non-null  object 
 4   production_value        12551 non-null  float64
 5   production_unit         12551 non-null  object 
 6   total_emissions_MtCO2e  12551 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 686.5+ KB
None
year                      0
parent_entity             0
parent_type               0
commodity                 0
production_value          0
production_unit           0
total_emissions_MtCO2e    0
dtype: int64



 **Dataset Details 1**


*  **Dataset Name :** emissions_high_granularity
*   **Source :** https://www.kaggle.com/datasets/joebeachcapital/carbon-majors-emissions-data/data
*   **File Format :** .csv


**Dataset Description**

The Carbon Majors Emissions Data contains historical greenhouse gas emissions attributed to major fossil fuel and cement producers worldwide. It combines operational and product-use emissions from investor-owned companies, state-owned enterprises, and nation-states. Each row represents the reported or estimated emissions for a specific entity in a given year.

**Feature Description**


*   Year : The reporting year of emissions data

*   Entity : Name of the company or nation-state responsible for emissions

*   Entity Type : Classification of the entity (Investor-Owned, State-Owned, Nation-State)


*   Country : Country where the entity is headquartered or operates
*   Scope 1 Emissions : Direct operational emissions (e.g., flaring, venting, methane leakage) in MtCO₂e

*   Scope 3 Emissions : Indirect emissions from combustion of sold products in MtCO₂e

*   Total Emissions : Combined Scope 1 and Scope 3 emissions in MtCO₂e

*   Cumulative Emissions : Running total of historical emissions attributed to the entity

*   Fuel Type : Type of fossil fuel or cement associated with emissions (Coal, Oil, Gas, Cement, etc.)



 **Dataset Details 2**


*  **Dataset Name :** emissions_low_granularity
*   **Source :** https://www.kaggle.com/datasets/joebeachcapital/carbon-majors-emissions-data/data
*   **File Format :** .csv


**Dataset Description**

The Carbon Majors Emissions Data contains historical greenhouse gas emissions attributed to major fossil fuel and cement producers worldwide. It combines operational and product-use emissions from investor-owned companies, state-owned enterprises, and nation-states. Each row represents the reported or estimated emissions for a specific entity in a given year.

**Feature Description**

*   YEAR : The reporting year of the data point

*   PARENT_ENTITY : The company, state-owned entity, or nation-state responsible for emissions

*   PARENT_TYPE : The type of entity – investor-owned company, state-owned entity, or nation-state

*   COMMODITY : The fuel/commodity associated with production (Oil, Natural Gas, Coal types, or Cement)

*   PRODUCTION_VALUE : The quantity of commodity produced in the reporting year

*   PRODUCTION_UNIT : The unit of measurement for production (Million barrels, Billion cubic feet, or Million tonnes depending on commodity)

*   SCOPE1_EMISSIONS_MtCO2e : Direct operational emissions from production activities (in million tonnes of CO₂ equivalent)

*   SCOPE3_EMISSIONS_MtCO2e : Indirect emissions from combustion/use of sold products (in million tonnes of CO₂ equivalent)

*   TOTAL_EMISSIONS_MtCO2e : The total greenhouse gas emissions attributed to the entity in that year (Scope 1 + Scope 3)

*   CUMULATIVE_EMISSIONS_MtCO2e : Running historical total of all emissions attributed to the entity up to that year


 **Dataset Details 3**


*  **Dataset Name :** emissions_medium_granularity
*   **Source :** https://www.kaggle.com/datasets/joebeachcapital/carbon-majors-emissions-data/data
*   **File Format :** .csv


**Dataset Description**

The Carbon Majors Emissions Data contains historical greenhouse gas emissions attributed to major fossil fuel and cement producers worldwide. It combines operational and product-use emissions from investor-owned companies, state-owned enterprises, and nation-states. Each row represents the reported or estimated emissions for a specific entity in a given year.

**Feature Description**


*   Year : The reporting year of emissions data

*   Entity : Name of the company or nation-state responsible for emissions

*   Entity Type : Classification of the entity (Investor-Owned, State-Owned, Nation-State)


*   Country : Country where the entity is headquartered or operates
*   Scope 1 Emissions : Direct operational emissions (e.g., flaring, venting, methane leakage) in MtCO₂e

*   Scope 3 Emissions : Indirect emissions from combustion of sold products in MtCO₂e

*   Total Emissions : Combined Scope 1 and Scope 3 emissions in MtCO₂e

*   Cumulative Emissions : Running total of historical emissions attributed to the entity

*   Fuel Type : Type of fossil fuel or cement associated with emissions (Coal, Oil, Gas, Cement, etc.)





## Loading the Dataset


In [22]:
import pandas as pd
df_high = pd.read_csv("/content/sample_data/emissions_high_granularity.csv")
df_low = pd.read_csv("/content/sample_data/emissions_high_granularity.csv")
df_medium = pd.read_csv("/content/sample_data/emissions_high_granularity.csv")

## Explore and Understand the Data

**HIGH GRANULARITY DATA**

In [23]:
df_high.head()
df_high.info()
df_high.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15797 entries, 0 to 15796
Data columns (total 16 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                15797 non-null  int64  
 1   parent_entity                       15797 non-null  object 
 2   parent_type                         15797 non-null  object 
 3   reporting_entity                    15797 non-null  object 
 4   commodity                           15797 non-null  object 
 5   production_value                    15797 non-null  float64
 6   production_unit                     15797 non-null  object 
 7   product_emissions_MtCO2             15797 non-null  float64
 8   flaring_emissions_MtCO2             15797 non-null  float64
 9   venting_emissions_MtCO2             15797 non-null  float64
 10  own_fuel_use_emissions_MtCO2        15797 non-null  float64
 11  fugitive_methane_emissions_MtCO2e   15797

,year,production_value,product_emissions_MtCO2,flaring_emissions_MtCO2,venting_emissions_MtCO2,own_fuel_use_emissions_MtCO2,fugitive_methane_emissions_MtCO2e,fugitive_methane_emissions_MtCH4,total_operational_emissions_MtCO2e,total_emissions_MtCO2e
count,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000
mean,1985.827942,327.879634,79.391514,0.517226,0.462462,0.688676,8.884203,0.317293,10.552566,89.944080
std,28.664256,1188.625001,261.984080,1.783744,1.804575,3.564171,31.358244,1.119937,34.790479,292.843491
min,1854.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1970.000000,11.800000,5.996490,0.000000,0.000000,0.000000,0.607068,0.021681,0.751999,7.208860
50%,1993.000000,59.970871,21.502409,0.015913,0.045247,0.000000,2.351126,0.083969,2.869611,25.116721
75%,2007.000000,246.375000,62.191954,0.197253,0.329719,0.162415,7.401655,0.264345,8.965620,72.255340
max,2022.000000,27192.000000,7769.222235,27.026872,41.458662,83.203465,877.683714,31.345847,877.683714,8646.905949


**LOW GRANULARITY DATA**

In [24]:
df_low.head()
df_low.info()
df_low.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15797 entries, 0 to 15796
Data columns (total 16 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                15797 non-null  int64  
 1   parent_entity                       15797 non-null  object 
 2   parent_type                         15797 non-null  object 
 3   reporting_entity                    15797 non-null  object 
 4   commodity                           15797 non-null  object 
 5   production_value                    15797 non-null  float64
 6   production_unit                     15797 non-null  object 
 7   product_emissions_MtCO2             15797 non-null  float64
 8   flaring_emissions_MtCO2             15797 non-null  float64
 9   venting_emissions_MtCO2             15797 non-null  float64
 10  own_fuel_use_emissions_MtCO2        15797 non-null  float64
 11  fugitive_methane_emissions_MtCO2e   15797

,year,production_value,product_emissions_MtCO2,flaring_emissions_MtCO2,venting_emissions_MtCO2,own_fuel_use_emissions_MtCO2,fugitive_methane_emissions_MtCO2e,fugitive_methane_emissions_MtCH4,total_operational_emissions_MtCO2e,total_emissions_MtCO2e
count,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000,15797.000000
mean,1985.827942,327.879634,79.391514,0.517226,0.462462,0.688676,8.884203,0.317293,10.552566,89.944080
std,28.664256,1188.625001,261.984080,1.783744,1.804575,3.564171,31.358244,1.119937,34.790479,292.843491
min,1854.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1970.000000,11.800000,5.996490,0.000000,0.000000,0.000000,0.607068,0.021681,0.751999,7.208860
50%,1993.000000,59.970871,21.502409,0.015913,0.045247,0.000000,2.351126,0.083969,2.869611,25.116721
75%,2007.000000,246.375000,62.191954,0.197253,0.329719,0.162415,7.401655,0.264345,8.965620,72.255340
max,2022.000000,27192.000000,7769.222235,27.026872,41.458662,83.203465,877.683714,31.345847,877.683714,8646.905949


**MEDIUM GRANULARITY DATA**

In [25]:
df_medium.head()
df_medium.info()
df_medium.describe()
df_medium.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15797 entries, 0 to 15796
Data columns (total 16 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                15797 non-null  int64  
 1   parent_entity                       15797 non-null  object 
 2   parent_type                         15797 non-null  object 
 3   reporting_entity                    15797 non-null  object 
 4   commodity                           15797 non-null  object 
 5   production_value                    15797 non-null  float64
 6   production_unit                     15797 non-null  object 
 7   product_emissions_MtCO2             15797 non-null  float64
 8   flaring_emissions_MtCO2             15797 non-null  float64
 9   venting_emissions_MtCO2             15797 non-null  float64
 10  own_fuel_use_emissions_MtCO2        15797 non-null  float64
 11  fugitive_methane_emissions_MtCO2e   15797

,0
year,0
parent_entity,0
parent_type,0
reporting_entity,0
commodity,0
production_value,0
production_unit,0
product_emissions_MtCO2,0
flaring_emissions_MtCO2,0
venting_emissions_MtCO2,0


## Data Preprocessing

**Column Normalization**

In [27]:
df_medium.columns = df_medium.columns.str.lower().str.replace(" ", "_")


**Handles Duplicates**

In [29]:
df_medium = df_medium.drop_duplicates()




**Data Type Conversion**

In [37]:
df_medium['year'] = df_medium['year'].astype(int)
df_medium['total_emissions'] = pd.to_numeric(df_medium['total_emissions'], errors='coerce')


KeyError: 'total_emissions'

**Missing Values**

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = ['scope_1_emissions','scope_3_emissions','total_emissions','cumulative_emissions']
cat_cols = ['entity','entity_type','country','fuel_type']

# Median for numeric
df_medium[num_cols] = df_medium[num_cols].fillna(df_medium[num_cols].median())

# Mode for categorical
for c in cat_cols:
    df_medium[c] = df_medium[c].fillna(df_medium[c].mode()[0])


**Feature Engineering – Policy Cap**



In [41]:

df_medium = df_medium.sort_values(["parent_entity","year"])
df_medium["rolling_med_3"] = df_medium.groupby("parent_entity")["total_emissions_mtco2e"].transform(lambda x: x.shift().rolling(3, min_periods=1).median())
df_medium["policy_cap"] = df_medium["rolling_med_3"].fillna(df_medium.groupby("year")["total_emissions_mtco2e"].transform("median")) * 0.95


**Traget Creation**

* Check Excess Emissions

In [43]:
df_medium["excess_over_cap"] = np.maximum(df_medium["total_emissions_mtco2e"] - df_medium["policy_cap"], 0)


* Action Label(Reduce or Buy)



In [44]:
df_medium["action"] = df_medium["excess_over_cap"].apply(lambda x: "reduce" if x>0 else "compliant")


**Final Feature Selection**

* Inputs (X): Numeric(scope1,scope3,year,etc.) + Categorical(entity_type,fuel_type)

* Targets:


*   Regression -> excess_over_cap
*   Classification -> action







In [49]:
# Numerical columns
num_cols = [
    "production_value",
    "product_emissions_mtco2",
    "flaring_emissions_mtco2",
    "venting_emissions_mtco2",
    "own_fuel_use_emissions_mtco2",
    "fugitive_methane_emissions_mtco2e",
    "fugitive_methane_emissions_mtch4",
    "total_operational_emissions_mtco2e",
    "total_emissions_mtco2e"
]

# Categorical columns
cat_cols = [
    "parent_entity",
    "parent_type",
    "reporting_entity",
    "commodity",
    "production_unit",
    "source"
]

# First, define a policy cap (this is a business assumption)
df_medium = df_medium.sort_values(["parent_entity", "year"])

df_medium["rolling_med_3"] = (
    df_medium.groupby("parent_entity")["total_emissions_mtco2e"]
    .transform(lambda x: x.shift().rolling(3, min_periods=1).median())
)

df_medium["policy_cap"] = (
    df_medium["rolling_med_3"].fillna(
        df_medium.groupby("year")["total_emissions_mtco2e"].transform("median")
    ) * 0.95
)

# Create target columns
df_medium["excess_over_cap"] = (
    df_medium["total_emissions_mtco2e"] - df_medium["policy_cap"]
)

df_medium["action"] = df_medium["excess_over_cap"].apply(
    lambda x: "reduce" if x > 0 else "ok"
)




## Data Splitting
Once preprocessing is complete, we divide the dataset into features (X) and targets (y), and then into train and test subsets.

**Define Features and Targets**

*   **Features (X):** emission values, year, fuel type, country, etc.
*   **Regression Target (y_reg):** excess_over_cap
* **Classification Target (y_cls):** action



In [48]:
# Features
X = df_medium[num_cols + cat_cols]

# Targets
y_reg = df_medium["excess_over_cap"]   # For regression
y_cls = df_medium["action"]            # For classification


**Train-Test Split (Regression)**

Split into 80% train and 20% test


In [50]:
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print("Regression → Train size:", X_train_reg.shape, "Test size:", X_test_reg.shape)


Regression → Train size: (12637, 15) Test size: (3160, 15)


**Train-Test Split (Classification)**

We’ll use the same X, but target changes to y_cls

In [51]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls  # stratify = keep class balance
)

print("Classification → Train size:", X_train_cls.shape, "Test size:", X_test_cls.shape)


Classification → Train size: (12637, 15) Test size: (3160, 15)
